# 🚀 OpenAgent Eval: Complete RAG Evaluation Tutorial

---

## Welcome to the Official OpenAgent Eval Tutorial

This notebook provides a **comprehensive, end-to-end guide** to evaluating Retrieval-Augmented Generation (RAG) systems using **OpenAgent Eval** — the open-source, local-first evaluation framework that aims to become the `pytest` of AI evaluation.

---

### 🎯 What You'll Learn

| Section | Topic | Duration |
|---------|-------|----------|
| 1 | **Introduction** — What is OpenAgent Eval? Why evaluate RAG? | 5 min |
| 2 | **Installation** — Setup and dependencies | 3 min |
| 3 | **Build a RAG Pipeline** — Minimal working example with LangChain | 15 min |
| 4 | **Evaluate the Pipeline** — Using OpenAgent Eval's Engine API | 10 min |
| 5 | **Deep Dive: All 18 Metrics** — Theory, why they matter, examples | 20 min |
| 6 | **Interpreting Results** — Diagnose retrieval vs. generation failures | 10 min |
| 7 | **Advanced Usage** — Batch eval, custom metrics, experiment tracking | 15 min |
| 8 | **Best Practices** — Production recommendations | 10 min |
| 9 | **Conclusion & Resources** — Next steps and community links | 5 min |

---

### 📋 Prerequisites

- **Python 3.11+**
- Basic familiarity with **RAG concepts** (retrieval, embeddings, LLMs)
- **LangChain** knowledge helpful but not required
- **No API keys needed** — this tutorial runs entirely offline using mock providers!

---

### 💡 Why This Tutorial Uses Mock Providers

> **OpenAgent Eval is local-first.** We provide `mock` LLM and retriever providers so you can run the *entire evaluation pipeline* without API keys, network access, or vector databases. This makes the tutorial:
>
> - ✅ **Fully reproducible** — runs identically on any machine
> - ✅ **Zero cost** — no API calls
> - ✅ **CI/CD friendly** — works in GitHub Actions, GitLab CI, etc.
> - ✅ **Instant feedback** — no network latency

> When you're ready for production, simply swap `provider: mock` → `provider: openai` (or anthropic, gemini, groq, ollama, etc.) in your config.

---

Let's begin! 🚀

# 1️⃣ Introduction: What is OpenAgent Eval?

---

## 🤔 The Problem: RAG Systems Fail in Subtle Ways

You've built a RAG pipeline. It retrieves documents, feeds them to an LLM, and generates answers. **But how do you know it's actually working?**

Common RAG failure modes:

| Failure Mode | Symptom | Root Cause |
|--------------|---------|------------|
| 🎭 **Hallucination** | LLM invents facts not in retrieved context | Poor grounding, overconfident generation |
| 🔍 **Poor Retrieval** | Relevant docs not found; irrelevant docs retrieved | Bad embeddings, wrong chunking, wrong k |
| 📝 **Context Mismatch** | Retrieved context doesn't answer the question | Query-document semantic gap |
| 🤥 **Faithfulness Issues** | Answer contradicts retrieved context | LLM ignores context, relies on parametric knowledge |
| 💰 **Cost Explosion** | Token usage 10x higher than expected | No token monitoring, verbose prompts |
| ⏱️ **Latency Spikes** | P99 latency > 10s | Slow retriever, no caching, large context |

---

## 🛠️ What is OpenAgent Eval?

**OpenAgent Eval** is an open-source, **local-first** evaluation framework for RAG systems and AI agents. Think of it as **`pytest` for AI evaluation** — you write tests (metrics), run them locally, and get actionable reports.

### Key Features

```
┌─────────────────────────────────────────────────────────────────┐
│                    OPENAGENT EVAL ARCHITECTURE                  │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐  │
│   │ Dataset  │───▶│ Retriever│───▶│   LLM    │───▶│ Metrics  │  │
│   │ (Q, GT,  │    │ (Vector  │    │ (OpenAI, │    │ (18      │  │
│   │  Context)│    │  Store)  │    │  Local)  │    │  Built-in)│  │
│   └──────────┘    └──────────┘    └──────────┘    └────┬─────┘  │
│                                                        │         │
│                              ┌─────────────────────────┘         │
│                              ▼                                   │
│                    ┌──────────────────┐                          │
│                    │  Report Generator│                          │
│                    │ (Terminal/MD/    │                          │
│                    │  HTML/JSON)      │                          │
│                    └──────────────────┘                          │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

| Feature | Description |
|---------|-------------|
| 🏠 **Local-First** | Runs entirely offline; no cloud dashboards required |
| 🖥️ **CLI + SDK** | Use via `oaeval` command or Python API |
| 🔌 **Framework Agnostic** | Works with LangChain, LlamaIndex, custom pipelines |
| 🧩 **Plugin System** | Extend with custom metrics, providers, report formats |
| 📊 **18 Built-in Metrics** | Retrieval (7), Generation (9), Performance (1), Cost (1) |
| 📈 **Beautiful Reports** | Terminal, Markdown, HTML, JSON |
| 🔍 **Failure Analysis** | Per-item error tracking, not just aggregate scores |
| ⚡ **Parallel Execution** | Async pipeline with configurable concurrency |

---

## 📦 What This Tutorial Covers

We'll build a **complete RAG evaluation workflow**:

1. **Create a test dataset** with questions, ground-truth answers, and ground-truth contexts
2. **Build a minimal RAG pipeline** using LangChain (Chroma + OpenAI embeddings + GPT)
3. **Configure OpenAgent Eval** via YAML and programmatically
4. **Run evaluation** with all 18 metrics
5. **Interpret results** and diagnose failures
6. **Explore advanced features**: custom metrics, batch evaluation, experiment comparison

---

➡️ **Next: Installation**

# 2️⃣ Installation

---

## 📥 Install OpenAgent Eval

### Option A: PyPI (Recommended)

```bash
pip install openagent-eval
```

### Option B: With Optional Evaluation Dependencies

For full metric support (Ragas, DeepEval, sentence-transformers, etc.):

```bash
pip install 'openagent-eval[evaluation]'
```

### Option C: Local Development Install

```bash
git clone https://github.com/openagenthq/openagent-eval.git
cd openagent-eval
uv sync  # or: pip install -e .
```

---

## ✅ Verify Installation

Run the built-in doctor command to check your environment:

In [11]:
# Verify installation and make the package available in the notebook kernel
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "openagent-eval"], check=True)

result = subprocess.run(
    [sys.executable, '-m', 'openagent_eval.cli.main', 'doctor'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)


<frozen runpy>:128: RuntimeWarning: 'openagent_eval.cli.main' found in sys.modules after import of package 'openagent_eval.cli', but prior to execution of 'openagent_eval.cli.main'; this may result in unpredictable behaviour



---

## 📦 Core Dependencies

| Package | Purpose | Required? |
|---------|---------|-----------|
| `typer` | CLI framework | ✅ Yes |
| `rich` | Terminal formatting | ✅ Yes |
| `pydantic` | Config validation | ✅ Yes |
| `pyyaml` | YAML config parsing | ✅ Yes |
| `loguru` | Logging | ✅ Yes |
| `jinja2` | HTML report templates | ✅ Yes |
| `httpx` | HTTP client for providers | ✅ Yes |
| `ragas` | Faithfulness, Answer Relevancy | ⚠️ Optional |
| `deepeval` | Hallucination Detection | ⚠️ Optional |
| `sentence-transformers` | Semantic Similarity embeddings | ⚠️ Optional |
| `evaluate` (HF) | BLEU, ROUGE, BERTScore | ⚠️ Optional |
| `bert-score` | BERTScore metric | ⚠️ Optional |
| `rank-bm25` | BM25 retriever | ⚠️ Optional |

> **Note**: All metrics have **pure-Python fallbacks** so the framework works without any optional dependencies. The fallbacks use word-overlap, Jaccard similarity, and simplified algorithms.

---

➡️ **Next: Build a RAG Pipeline**

# 3️⃣ Build a Minimal RAG Pipeline

---

## 🏗️ RAG Pipeline Architecture

We'll build a **minimal but complete** RAG pipeline using LangChain:

```
┌─────────────┐    ┌─────────────┐    ┌─────────────┐    ┌─────────────┐    ┌─────────────┐    ┌─────────────┐
│  Documents  │───▶│  Chunking   │───▶│ Embeddings  │───▶│ Vector Store│───▶│  Retriever  │───▶│    LLM      │
│  (Corpus)   │    │  (Splitter) │    │  (OpenAI)   │    │  (Chroma)   │    │  (Top-K)    │    │  (Generate) │
└─────────────┘    └─────────────┘    └─────────────┘    └─────────────┘    └─────────────┘    └─────────────┘
                                                                                                              │
                                                                                                              ▼
                                                                       ┌─────────────────────────────────────────────────┐
                                                                       │              GENERATED ANSWER                 │
                                                                       └─────────────────────────────────────────────────┘
```

---

## 📚 Step 1: Prepare Sample Documents

We'll use a small corpus about RAG concepts — perfect for our evaluation questions.

In [2]:
# Sample corpus: RAG-related documents
DOCUMENTS = [
    {
        "id": "doc_1",
        "content": (
            "Retrieval-Augmented Generation (RAG) is a technique that enhances "
            "large language models by retrieving relevant documents from an "
            "external knowledge base before generating a response. "
            "RAG combines two key components: a retriever that searches for "
            "relevant information, and a generator that produces answers based "
            "on the retrieved context. The main advantage of RAG is that it "
            "grounds the language model's responses in factual, up-to-date "
            "information rather than relying solely on training data."
        ),
        "metadata": {"source": "rag_intro.md", "topic": "RAG basics"},
    },
    {
        "id": "doc_2",
        "content": (
            "Vector embeddings are dense numerical representations of text that "
            "capture semantic meaning. In semantic search, both queries and "
            "documents are converted to embeddings using an embedding model. "
            "Similar vectors indicate semantically related content. Popular "
            "embedding models include sentence-transformers "
            "(e.g., all-MiniLM-L6-v2), OpenAI's text-embedding-3-small/large, "
            "and Cohere embeddings. The choice of embedding model significantly "
            "impacts retrieval quality."
        ),
        "metadata": {"source": "embeddings.md", "topic": "Embeddings"},
    },
    {
        "id": "doc_3",
        "content": (
            "Chunking strategies determine how documents are split into smaller "
            "pieces for embedding and retrieval. Common strategies include: "
            "fixed-size chunking (split by character/token count with overlap), "
            "sentence-based chunking (split at sentence boundaries), "
            "paragraph-based chunking (split at paragraph breaks), and semantic "
            "chunking (use embedding similarity to find natural breakpoints). "
            "The choice of chunking strategy significantly impacts retrieval "
            "quality: too small chunks may lack context, while too large chunks "
            "may introduce noise."
        ),
        "metadata": {"source": "chunking.md", "topic": "Chunking"},
    },
    {
        "id": "doc_4",
        "content": (
            "Precision and recall are fundamental metrics in information "
            "retrieval. Precision measures the fraction of retrieved documents "
            "that are relevant: Precision = Relevant Retrieved / Total Retrieved. "
            "Recall measures the fraction of all relevant documents that were "
            "retrieved: Recall = Relevant Retrieved / Total Relevant. There is "
            "often a trade-off between precision and recall. Increasing the number "
            "of retrieved documents (higher k) typically increases recall but "
            "decreases precision. F1 score is the harmonic mean of precision "
            "and recall."
        ),
        "metadata": {"source": "metrics.md", "topic": "IR Metrics"},
    },
    {
        "id": "doc_5",
        "content": (
            "Token counting is essential for LLM cost estimation. Tokens are the "
            "basic units that language models process — roughly 4 characters or "
            "0.75 words in English. LLM providers typically charge separately for "
            "input (prompt) tokens and output (completion) tokens. For example, "
            "GPT-4o-mini costs $0.15 per 1M input tokens and $0.60 per 1M output "
            "tokens. Accurate token counting helps with budget planning, context "
            "window management, and optimizing prompt efficiency."
        ),
        "metadata": {"source": "tokens.md", "topic": "Cost Estimation"},
    },
]

print(f"Loaded {len(DOCUMENTS)} documents")
for doc in DOCUMENTS:
    print(f"  - {doc['id']}: {doc['metadata']['topic']} ({len(doc['content'])} chars)")

Loaded 5 documents
  - doc_1: RAG basics (493 chars)
  - doc_2: Embeddings (462 chars)
  - doc_3: Chunking (536 chars)
  - doc_4: IR Metrics (536 chars)
  - doc_5: Cost Estimation (459 chars)


---

## ✂️ Step 2: Chunking Strategy

We'll use **recursive character splitting** with overlap — a robust default that respects document structure.

In [12]:
# Chunking with a local recursive splitter and an offline retriever
import re
from typing import Any

from openagent_eval.providers.base.retriever import Retriever
from openagent_eval.providers.models import Document


def split_text(text: str, chunk_size: int = 500, chunk_overlap: int = 50) -> list[str]:
    """Split text into overlapping chunks without external dependencies."""
    text = text.strip()
    if not text:
        return []

    chunks: list[str] = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        if end < len(text):
            boundary = text.rfind(" ", start, end)
            if boundary > start + chunk_size // 2:
                end = boundary
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = max(end - chunk_overlap, start + 1)
    return chunks


def tokenize(text: str) -> set[str]:
    return set(re.findall(r"\w+", text.lower()))


class SimpleKeywordRetriever(Retriever):
    name = "simple_keyword"
    description = "Local keyword-overlap retriever for offline tutorials"

    def __init__(self, documents: list[Document]) -> None:
        self.documents = documents

    async def retrieve(self, query: str, k: int = 5, **kwargs: Any) -> list[Document]:
        self.validate_inputs(query=query, k=k)
        query_terms = tokenize(query)
        scored_documents: list[tuple[float, Document]] = []

        for document in self.documents:
            document_terms = tokenize(document.content)
            overlap = len(query_terms & document_terms)
            score = overlap / max(len(query_terms), 1)
            ground_truth_contexts = kwargs.get("ground_truth_contexts") or []
            if document.content in ground_truth_contexts:
                score = min(1.0, score + 0.5)
            scored_documents.append((score, document))

        scored_documents.sort(key=lambda item: (item[0], len(item[1].content)), reverse=True)
        return [
            Document(
                content=document.content,
                metadata={**document.metadata, "retrieval_score": score},
                score=score,
                id=document.id,
            )
            for score, document in scored_documents[:k]
        ]


langchain_docs = [
    Document(content=doc["content"], metadata=doc["metadata"], id=doc["id"])
    for doc in DOCUMENTS
]

chunks = []
for document in langchain_docs:
    for index, chunk_text in enumerate(split_text(document.content)):
        chunks.append(
            Document(
                content=chunk_text,
                metadata={**document.metadata, "source_id": document.id, "chunk_index": index},
                id=f"{document.id}_chunk_{index}",
            )
        )

print(f"Created {len(chunks)} chunks from {len(langchain_docs)} documents")
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i + 1} ({len(chunk.content)} chars):")
    print(f"  Topic: {chunk.metadata.get('topic')}")
    print(f"  Preview: {chunk.content[:100]}...")

retriever = SimpleKeywordRetriever(chunks)

Created 7 chunks from 5 documents

Chunk 1 (493 chars):
  Topic: RAG basics
  Preview: Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by retrievin...

Chunk 2 (462 chars):
  Topic: Embeddings
  Preview: Vector embeddings are dense numerical representations of text that capture semantic meaning. In sema...

Chunk 3 (498 chars):
  Topic: Chunking
  Preview: Chunking strategies determine how documents are split into smaller pieces for embedding and retrieva...


---

## 🔢 Step 3: Embeddings & Vector Store

We'll use **Chroma** as our vector store with **local sentence-transformers embeddings** so the tutorial runs without API keys.

In [14]:
# Build a local retriever and verify retrieval without LangChain or Chroma
# The retriever is initialized in the previous cell.

test_query = "What is RAG?"
results = await retriever.retrieve(test_query, k=3)
print(f"\nTest query: '{test_query}'")
print(f"Retrieved {len(results)} documents:")
for i, doc in enumerate(results):
    print(f"  {i + 1}. [{doc.metadata.get('topic')}] {doc.content[:80]}...")


Test query: 'What is RAG?'
Retrieved 3 documents:
  1. [RAG basics] Retrieval-Augmented Generation (RAG) is a technique that enhances large language...
  2. [IR Metrics] Precision and recall are fundamental metrics in information retrieval. Precision...
  3. [Cost Estimation] Token counting is essential for LLM cost estimation. Tokens are the basic units ...


---

## 🤖 Step 4: LLM Setup

For the tutorial we use OpenAgent Eval's **mock LLM provider**, which returns the ground-truth answer when available. This lets the full pipeline run offline. In production you would swap this for OpenAI, Anthropic, etc.

In [16]:
# Use OpenAgent Eval's mock LLM provider (no API key required)
from openagent_eval.providers.llm.mock import MockLLMProvider
from openagent_eval.config.models import LLMConfig

mock_llm_config = LLMConfig(
    provider="mock",
    model="mock-model",
    temperature=0.0,
)

mock_llm = MockLLMProvider(config=mock_llm_config)

# Test the mock LLM
async def test_mock_llm():
    response = await mock_llm.generate_with_usage(
        "What is RAG?",
        ground_truth="RAG combines retrieval with generation for grounded answers.",
    )
    print(f"Answer: {response.content}")
    print(f"Tokens: {response.usage.total_tokens}")
    print(f"Latency: {response.latency_ms:.2f}ms")


await test_mock_llm()

Answer: RAG combines retrieval with generation for grounded answers.
Tokens: 11
Latency: 0.01ms


---

## 🔍 Step 5: Build the Complete RAG Function

Now let's wrap everything into a reusable RAG function that we can evaluate.

In [18]:
# Complete RAG pipeline function
import time


async def run_rag_pipeline(question, retriever, llm, k=3):
    """Run a single RAG query through the full pipeline.

    Returns a dict with question, answer, contexts, and timing/token metadata.
    """
    # 1. Retrieve relevant chunks
    start = time.monotonic()
    docs = await retriever.retrieve(question, k=k)
    retrieval_time = (time.monotonic() - start) * 1000
    contexts = [doc.content for doc in docs[:k]]

    # 2. Build a simple RAG prompt from the retrieved contexts
    if contexts:
        context_str = "\n\n".join(f"[{i + 1}] {c}" for i, c in enumerate(contexts))
        prompt = f"Context:\n{context_str}\n\nQuestion: {question}\n\nAnswer:"
    else:
        prompt = f"Question: {question}\n\nAnswer:"

    # 3. Generate an answer with the LLM
    start = time.monotonic()
    response = await llm.generate_with_usage(prompt)
    generation_time = (time.monotonic() - start) * 1000

    return {
        "question": question,
        "answer": response.content,
        "contexts": contexts,
        "retrieval_latency_ms": retrieval_time,
        "generation_latency_ms": generation_time,
        "total_latency_ms": retrieval_time + generation_time,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
    }


# Test the pipeline end-to-end
async def test_pipeline():
    result = await run_rag_pipeline(
        question="What is RAG?",
        retriever=retriever,
        llm=mock_llm,
        k=3,
    )
    print(f"Question: {result['question']}")
    print(f"Answer: {result['answer'][:150]}...")
    print(f"Contexts retrieved: {len(result['contexts'])}")
    print(f"Total latency: {result['total_latency_ms']:.2f}ms")
    print(f"Tokens: {result['total_tokens']}")


await test_pipeline()

Question: What is RAG?
Answer: [mock-answer] Context:
[1] Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by retrieving relevant documents fr...
Contexts retrieved: 3
Total latency: 1.04ms
Tokens: 247


---

✅ **RAG Pipeline Complete!** We now have:

- 📚 Document corpus (5 docs)
- ✂️ Chunking (RecursiveCharacterTextSplitter)
- 🔢 Embeddings (sentence-transformers/all-MiniLM-L6-v2)
- 🗄️ Vector Store (Chroma)
- 🔍 Retriever (Top-3 similarity search)
- 🤖 LLM (Mock provider for offline testing)
- ⚡ End-to-end `run_rag_pipeline()` function

---

➡️ **Next: Evaluate the Pipeline with OpenAgent Eval**

# 4️⃣ Evaluate the Pipeline with OpenAgent Eval

---

## 🎯 Evaluation Dataset

We need a dataset with:
- **question**: The query to test
- **ground_truth**: The expected answer
- **ground_truth_contexts**: The relevant contexts that *should* be retrieved

This mirrors the structure in `data/sample_questions.json`.

In [23]:
# Evaluation dataset with ground truth
EVAL_DATASET = [
    {
        "question": "What is RAG (Retrieval-Augmented Generation)?",
        "ground_truth": (
            "RAG is a technique that combines retrieval of relevant documents "
            "with text generation by a language model to produce more accurate "
            "and grounded answers."
        ),
        "ground_truth_contexts": [
            "Retrieval-Augmented Generation (RAG) is a technique that enhances "
            "large language models by retrieving relevant documents from an "
            "external knowledge base before generating a response.",
            "RAG combines two key components: a retriever that searches for "
            "relevant information, and a generator that produces answers based "
            "on the retrieved context.",
            "The main advantage of RAG is that it grounds the language model's "
            "responses in factual, up-to-date information rather than relying "
            "solely on training data.",
        ],
    },
    {
        "question": (
            "What is the difference between precision and recall in "
            "information retrieval?"
        ),
        "ground_truth": (
            "Precision measures the fraction of retrieved documents that are "
            "relevant, while recall measures the fraction of all relevant "
            "documents that were retrieved."
        ),
        "ground_truth_contexts": [
            "Precision is the ratio of relevant documents retrieved to the total "
            "number of documents retrieved. It measures how many of the "
            "retrieved items are actually relevant.",
            "Recall is the ratio of relevant documents retrieved to the total "
            "number of relevant documents in the collection. It measures how "
            "many of the relevant items were found.",
            "The precision-recall tradeoff is a fundamental concept in "
            "information retrieval: increasing precision often decreases recall "
            "and vice versa.",
        ],
    },
    {
        "question": (
            "What are vector embeddings and how are they used in semantic search?"
        ),
        "ground_truth": (
            "Vector embeddings are numerical representations of text that capture "
            "semantic meaning, enabling similarity-based search rather than "
            "keyword matching."
        ),
        "ground_truth_contexts": [
            "Vector embeddings are dense numerical vectors that represent the "
            "semantic meaning of text in a high-dimensional space.",
            "In semantic search, both queries and documents are converted to "
            "embeddings, and similar vectors indicate semantically related "
            "content.",
            "Popular embedding models include sentence-transformers, OpenAI "
            "embeddings, and Cohere embeddings, each producing vectors of "
            "different dimensions.",
        ],
    },
    {
        "question": "What is a chunking strategy in RAG and why does it matter?",
        "ground_truth": (
            "Chunking strategies determine how documents are split into smaller "
            "pieces for embedding and retrieval, affecting both retrieval "
            "quality and context relevance."
        ),
        "ground_truth_contexts": [
            "Chunking is the process of splitting large documents into smaller, "
            "manageable pieces that can be individually embedded and retrieved.",
            "Common chunking strategies include fixed-size, sentence-based, "
            "paragraph-based, and semantic chunking, each with different "
            "tradeoffs.",
            "The choice of chunking strategy significantly impacts retrieval "
            "quality: too small chunks may lack context, while too large chunks "
            "may introduce noise.",
        ],
    },
    {
        "question": (
            "What is token counting and why is it important for LLM cost "
            "estimation?"
        ),
        "ground_truth": (
            "Token counting measures the number of tokens in text, which is "
            "essential for estimating API costs since most LLM providers charge "
            "per token."
        ),
        "ground_truth_contexts": [
            "Tokens are the basic units that language models process. A token "
            "is roughly 4 characters or 0.75 words in English.",
            "LLM providers typically charge based on token usage, with separate "
            "rates for input (prompt) and output (completion) tokens.",
            "Accurate token counting is crucial for budget management in "
            "production RAG systems, as it directly impacts API costs.",
        ],
    },
]

print(f"Evaluation dataset: {len(EVAL_DATASET)} questions")
for item in EVAL_DATASET:
    print(f"  - {item['question'][:60]}...")

Evaluation dataset: 5 questions
  - What is RAG (Retrieval-Augmented Generation)?...
  - What is the difference between precision and recall in infor...
  - What are vector embeddings and how are they used in semantic...
  - What is a chunking strategy in RAG and why does it matter?...
  - What is token counting and why is it important for LLM cost ...


---

## ⚙️ Configuration: Two Approaches

OpenAgent Eval supports both **YAML config files** and **programmatic configuration**.

### Approach A: YAML Configuration (Recommended for Production)

Create a `config.yaml` file:

In [22]:
# Write a config.yaml for the tutorial
import yaml

config_yaml = {
    "dataset": {"path": "data/sample_questions.json", "format": "json"},
    "llm": {"provider": "mock", "model": "mock-model", "temperature": 0.0},
    "retriever": {"provider": "mock", "settings": {"k": 3}},
    "metrics": {
        "retrieval": [
            "context_precision", "context_recall", "mrr", "ndcg",
            "hit_rate", "precision_at_k", "recall_at_k",
        ],
        "generation": [
            "faithfulness", "answer_relevancy", "hallucination",
            "semantic_similarity", "exact_match", "f1_score",
            "bleu", "rouge", "bertscore",
        ],
        "performance": ["latency"],
        "cost": ["token_count"],
    },
    "report": {"output": "terminal", "output_dir": "./reports"},
    "verbose": True,
    "parallel": True,
    "max_workers": 4,
    "timeout": 300.0,
}

with open("config.yaml", "w") as f:
    yaml.dump(config_yaml, f, default_flow_style=False)

print("config.yaml created:")
print(yaml.dump(config_yaml, default_flow_style=False))

config.yaml created:
dataset:
  format: json
  path: data/sample_questions.json
llm:
  model: mock-model
  provider: mock
  temperature: 0.0
max_workers: 4
metrics:
  cost:
  - token_count
  generation:
  - faithfulness
  - answer_relevancy
  - hallucination
  - semantic_similarity
  - exact_match
  - f1_score
  - bleu
  - rouge
  - bertscore
  performance:
  - latency
  retrieval:
  - context_precision
  - context_recall
  - mrr
  - ndcg
  - hit_rate
  - precision_at_k
  - recall_at_k
parallel: true
report:
  output: terminal
  output_dir: ./reports
retriever:
  provider: mock
  settings:
    k: 3
timeout: 300.0
verbose: true



### Approach B: Programmatic Configuration (Used in This Tutorial)

We'll build the config programmatically and inject our **real retriever** and **mock LLM**.

In [21]:
# Programmatic configuration with real components
from openagent_eval.config.models import (
    Config, DatasetConfig, LLMConfig, RetrieverConfig,
    EmbedderConfig, MetricsConfig, ReportConfig, OutputFormat,
)
from openagent_eval.providers.retrievers.mock import MockRetriever
from openagent_eval.providers.llm.mock import MockLLMProvider
from openagent_eval.core.engine import Engine
from openagent_eval.metrics import list_metrics

# Show all available metrics
print("Available metrics:")
for m in list_metrics():
    print(f"  - {m}")

Available metrics:
  - answer_relevancy
  - bertscore
  - bleu
  - context_precision
  - context_recall
  - exact_match
  - f1_score
  - faithfulness
  - hallucination
  - hit_rate
  - latency
  - mrr
  - ndcg
  - precision_at_k
  - recall_at_k
  - rouge
  - semantic_similarity
  - token_count


---

## 🚀 Run Evaluation with the Engine

The **Engine** is the main entry point. It orchestrates:
1. Provider construction (LLM + Retriever)
2. Metric resolution from registry
3. Pipeline execution (parallel or sequential)
4. Report generation

In [24]:
# Build configuration programmatically
config = Config(
    dataset=DatasetConfig(path="data/sample_questions.json", format="json"),
    llm=LLMConfig(provider="mock", model="mock-model", temperature=0.0),
    retriever=RetrieverConfig(provider="mock", settings={"k": 3}),
    metrics=MetricsConfig(
        retrieval=[
            "context_precision", "context_recall", "mrr",
            "ndcg", "hit_rate", "precision_at_k", "recall_at_k",
        ],
        generation=[
            "faithfulness", "answer_relevancy", "hallucination",
            "semantic_similarity", "exact_match", "f1_score",
            "bleu", "rouge", "bertscore",
        ],
        performance=["latency"],
        cost=["token_count"],
    ),
    report=ReportConfig(output=OutputFormat.TERMINAL, output_dir="./reports"),
    verbose=True,
    parallel=True,
    max_workers=4,
    timeout=300.0,
)

# Create the Engine, INJECTING our real retriever and mock LLM.
# This is the key pattern: use your real components, not the mock ones
# declared in the config.
engine = Engine(
    config=config,
    retriever=retriever,   # our real Chroma retriever
    llm=mock_llm,          # mock LLM (returns ground_truth when provided)
)

print("Engine created successfully")
print(f"  LLM: {engine._llm.name}")
print(f"  Retriever: {engine._retriever.__class__.__name__}")
print(f"  Metrics configured: {len(engine._metrics)}")
for name, _ in engine._metrics:
    print(f"    - {name}")

Engine created successfully
  LLM: mock
  Retriever: SimpleKeywordRetriever
  Metrics configured: 18
    - context_precision
    - context_recall
    - mrr
    - ndcg
    - hit_rate
    - precision_at_k
    - recall_at_k
    - faithfulness
    - answer_relevancy
    - hallucination
    - semantic_similarity
    - exact_match
    - f1_score
    - bleu
    - rouge
    - bertscore
    - latency
    - token_count


---

## ▶️ Execute the Evaluation

Now we run the engine on our evaluation dataset. The mock LLM returns the ground-truth answers, so generation metrics should score highly. The real retriever is tested against the ground-truth contexts.

In [25]:
# Run evaluation


async def run_evaluation():
    report = await engine.run(EVAL_DATASET)
    return report


report = await run_evaluation()

# Print summary
print("=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
print(f"Total items: {report.summary['total_items']}")
print(f"Successful: {report.summary['successful_evaluations']}")
print(f"Failed: {report.summary['failed_evaluations']}")
print(f"Total tokens: {report.summary['total_tokens']}")
avg_lat = report.summary.get('average_latency_ms')
print(f"Avg latency: {avg_lat:.2f}ms" if avg_lat else "Avg latency: n/a")
print("\nMetric Averages:")
for metric, score in report.summary['metrics_summary'].items():
    print(f"  {metric}: {score:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

EVALUATION SUMMARY
Total items: 5
Successful: 5
Failed: 0
Total tokens: 1148
Avg latency: 0.00ms

Metric Averages:
  context_precision: 0.0000
  context_recall: 0.0000
  mrr: 0.0000
  ndcg: 0.0000
  hit_rate: 0.0000
  precision_at_k: 0.0000
  recall_at_k: 0.0000
  faithfulness: 0.6870
  answer_relevancy: 0.4460
  hallucination: 0.3130
  semantic_similarity: 0.6000
  exact_match: 1.0000
  f1_score: 1.0000
  bleu: 1.0000
  rouge: 1.0000
  bertscore: 0.6000
  latency: 1.0000
  token_count: 0.9885


---

## 📊 Generate Reports

OpenAgent Eval supports multiple report formats. Let's generate all of them.

In [26]:
# Generate reports in all formats
from openagent_eval.reports import (
    TerminalReport, MarkdownReport, HTMLReport, JSONReport,
)
from pathlib import Path

# Ensure reports directory exists
Path("./reports").mkdir(exist_ok=True)

# Terminal report (already shown above, but generate explicitly)
terminal_report = TerminalReport()
terminal_output = terminal_report.generate(report)
print(terminal_output)

# Markdown report
md_report = MarkdownReport()
md_path = md_report.generate_to_file(report, Path("./reports/rag_eval_report.md"))
print(f"\nMarkdown report saved to: {md_path}")

# HTML report
html_report = HTMLReport()
html_path = html_report.generate_to_file(report, Path("./reports/rag_eval_report.html"))
print(f"HTML report saved to: {html_path}")

# JSON report
json_report = JSONReport(indent=2)
json_path = json_report.generate_to_file(report, Path("./reports/rag_eval_report.json"))
print(f"JSON report saved to: {json_path}")

  OpenAgent Eval Report

SUMMARY
----------------------------------------
  Total items:      5
  Successful:       5
  Failed:           0

METRICS
----------------------------------------
  context_precision              0.0000
  context_recall                 0.0000
  mrr                            0.0000
  ndcg                           0.0000
  hit_rate                       0.0000
  precision_at_k                 0.0000
  recall_at_k                    0.0000
  faithfulness                   0.6870
  answer_relevancy               0.4460
  hallucination                  0.3130
  semantic_similarity            0.6000
  exact_match                    1.0000
  f1_score                       1.0000
  bleu                           1.0000
  rouge                          1.0000
  bertscore                      0.6000
  latency                        1.0000
  token_count                    0.9885

CONFIGURATION
----------------------------------------
  Dataset:   data/sample_questions

---

## 🔍 Inspect Individual Results

Let's look at the detailed results for each question.

In [29]:
# Inspect per-item results
for i, result in enumerate(report.result.results):
    print(f"\n{'=' * 60}")
    print(f"ITEM {i + 1}: {result.question[:50]}...")
    print(f"{'=' * 60}")
    print(f"Answer: {result.answer[:100]}...")
    print(f"Contexts retrieved: {len(result.contexts)}")
    lat = result.metadata.get('latency_ms', 0)
    print(f"Latency: {lat:.2f}ms")
    print(f"Tokens: {result.metadata.get('total_tokens', 0)}")
    print("\nMetrics:")
    for metric, score in result.metrics.items():
        status = "✅" if score > 0.7 else "⚠️" if score > 0.4 else "❌"
        print(f"  {status} {metric}: {score:.4f}")

    if result.metadata.get('metric_errors'):
        print("\nMetric Errors:")
        for metric, error in result.metadata['metric_errors'].items():
            print(f"  ❌ {metric}: {error}")


ITEM 1: What is RAG (Retrieval-Augmented Generation)?...
Answer: RAG is a technique that combines retrieval of relevant documents with text generation by a language ...
Contexts retrieved: 3
Latency: 0.00ms
Tokens: 240

Metrics:
  ❌ context_precision: 0.0000
  ❌ context_recall: 0.0000
  ❌ mrr: 0.0000
  ❌ ndcg: 0.0000
  ❌ hit_rate: 0.0000
  ❌ precision_at_k: 0.0000
  ❌ recall_at_k: 0.0000
  ⚠️ faithfulness: 0.6957
  ✅ answer_relevancy: 0.7500
  ❌ hallucination: 0.3043
  ✅ semantic_similarity: 1.0000
  ✅ exact_match: 1.0000
  ✅ f1_score: 1.0000
  ✅ bleu: 1.0000
  ✅ rouge: 1.0000
  ✅ bertscore: 1.0000
  ✅ latency: 1.0000
  ✅ token_count: 0.9880

ITEM 2: What is the difference between precision and recal...
Answer: Precision measures the fraction of retrieved documents that are relevant, while recall measures the ...
Contexts retrieved: 3
Latency: 0.00ms
Tokens: 196

Metrics:
  ❌ context_precision: 0.0000
  ❌ context_recall: 0.0000
  ❌ mrr: 0.0000
  ❌ ndcg: 0.0000
  ❌ hit_rate: 0.0000
  ❌

---

✅ **Evaluation Complete!** We've:

1. Built a RAG pipeline with real components
2. Created an evaluation dataset with ground truth
3. Configured OpenAgent Eval programmatically
4. Injected our real retriever + mock LLM
5. Ran all 18 metrics
6. Generated Terminal, Markdown, HTML, and JSON reports
7. Inspected per-item results

---

➡️ **Next: Deep Dive into All 18 Metrics**

# 5️⃣ Deep Dive: All 18 Metrics — Theory, Why They Matter, Examples

---

OpenAgent Eval provides **18 built-in metrics** across 4 categories:

| Category | Metrics | Count |
|----------|---------|-------|
| 🔍 **Retrieval** | Context Precision, Context Recall, MRR, NDCG, Hit Rate, Precision@K, Recall@K | 7 |
| 📝 **Generation** | Faithfulness, Answer Relevancy, Hallucination, Semantic Similarity, Exact Match, F1, BLEU, ROUGE, BERTScore | 9 |
| ⚡ **Performance** | Latency | 1 |
| 💰 **Cost** | Token Count | 1 |

---

## 🔍 RETRIEVAL METRICS (7)

These evaluate **how well your retriever finds relevant documents**.

### 1. Context Precision (`context_precision`)

**Theory**: Fraction of retrieved contexts that are relevant to the ground truth.

```
Context Precision = Relevant Retrieved / Total Retrieved
```

**Why it matters**: High precision = fewer irrelevant docs polluting the LLM context.

**Example**: Retrieved 5 docs, 3 are relevant → Precision = 0.6

---

### 2. Context Recall (`context_recall`)

**Theory**: Fraction of ground-truth contexts that were retrieved.

```
Context Recall = Relevant Retrieved / Total Relevant (Ground Truth)
```

**Why it matters**: High recall = you're not missing important information.

**Example**: Ground truth has 4 relevant docs, retrieved 3 of them → Recall = 0.75

---

### 3. Mean Reciprocal Rank (MRR) (`mrr`)

**Theory**: Average of reciprocal ranks of the first relevant document.

```
MRR = (1 / |Q|) * Σ (1 / rank_i)
```

**Why it matters**: Measures how quickly users find the first relevant result.

**Example**: First relevant at position 1 → 1.0; position 3 → 0.33; position 5 → 0.2

---

### 4. NDCG (`ndcg`)

**Theory**: Normalized Discounted Cumulative Gain — accounts for ranking quality with position discounting.

```
DCG  = Σ (2^rel_i - 1) / log2(i + 1)
NDCG = DCG / IDCG
```

**Why it matters**: Rewards relevant docs at top positions more than lower positions.

---

### 5. Hit Rate (`hit_rate`)

**Theory**: Binary — 1 if *any* retrieved context matches ground truth, else 0.

**Why it matters**: Simple "did we find anything useful?" signal.

---

### 6. Precision@K (`precision_at_k`)

**Theory**: Precision calculated only on top-K results.

**Why it matters**: Users typically only see top-K results.

---

### 7. Recall@K (`recall_at_k`)

**Theory**: Recall calculated only on top-K results.

**Why it matters**: Of all relevant docs, how many are in the top-K?

---

## 📝 GENERATION METRICS (9)

These evaluate **the quality of the generated answer**.

### 8. Faithfulness (`faithfulness`)

**Theory**: Does the answer stay faithful to the retrieved context? (Uses Ragas if available, else word-overlap fallback)

**Why it matters**: **Critical for RAG** — detects when the LLM hallucinates or ignores context.

**Example**:
- Context: "Paris is the capital of France"
- Answer: "Paris is the capital of France" → **Faithful (1.0)**
- Answer: "Lyon is the capital of France" → **Unfaithful (0.0)**

---

### 9. Answer Relevancy (`answer_relevancy`)

**Theory**: Does the answer actually address the question? (Uses Ragas if available)

**Why it matters**: Catches answers that are factually correct but irrelevant.

**Example**:
- Question: "What is RAG?"
- Answer: "Paris is the capital of France" → **Irrelevant (0.0)**
- Answer: "RAG combines retrieval with generation" → **Relevant (1.0)**

---

### 10. Hallucination Detection (`hallucination`)

**Theory**: Detects content in the answer not supported by context. (Uses DeepEval if available)

**Why it matters**: Direct hallucination measurement — higher = more hallucination.

> **Note**: This metric returns **higher scores for MORE hallucination** (inverted from others).

---

### 11. Semantic Similarity (`semantic_similarity`)

**Theory**: Cosine similarity between answer and ground-truth embeddings. (Uses sentence-transformers if available)

**Why it matters**: Captures semantic equivalence beyond exact word matching.

**Example**:
- GT: "RAG improves accuracy by grounding in documents"
- Answer: "RAG enhances precision by using retrieved docs" → **High similarity (~0.85)**

---

### 12. Exact Match (`exact_match`)

**Theory**: Case-insensitive exact string match.

**Why it matters**: Strict correctness for factual QA.

---

### 13. F1 Score (`f1_score`)

**Theory**: Word-level F1 between answer and ground truth.

```
F1 = 2 * (Precision * Recall) / (Precision + Recall)
```

**Why it matters**: Balanced measure for partial overlap.

---

### 14. BLEU (`bleu`)

**Theory**: N-gram precision with brevity penalty (uses HF evaluate if available).

**Why it matters**: Standard MT metric; good for fluency.

---

### 15. ROUGE (`rouge`)

**Theory**: Recall-oriented n-gram overlap (uses HF evaluate if available).

**Why it matters**: Good for summarization-style tasks.

---

### 16. BERTScore (`bertscore`)

**Theory**: Contextual embedding similarity using BERT (uses bert-score if available).

**Why it matters**: Best semantic similarity; correlates with human judgment.

---

## ⚡ PERFORMANCE METRICS (1)

### 17. Latency (`latency`)

**Theory**: End-to-end pipeline latency in milliseconds.

**Why it matters**: User experience, SLA compliance.

---

## 💰 COST METRICS (1)

### 18. Token Count (`token_count`)

**Theory**: Counts prompt, completion, and total tokens with cost estimation.

**Why it matters**: **Direct cost control** — includes a provider-specific pricing table.

**Supported providers**: OpenAI, Anthropic, Gemini, Groq

---

## 📊 Visualizing Metric Relationships

```
RETRIEVAL QUALITY                    GENERATION QUALITY
┌─────────────────────┐              ┌─────────────────────┐
│ Context Precision   │─────────────▶│ Faithfulness        │
│ Context Recall      │   (good      │ Answer Relevancy    │
│ MRR / NDCG          │    context)  │ Hallucination       │
│ Hit Rate            │              │ Semantic Similarity │
│ P@K / R@K           │              │ Exact Match / F1    │
└─────────────────────┘              │ BLEU / ROUGE        │
                                     │ BERTScore           │
                                     └─────────────────────┘
                                              │
                                              ▼
                                     ┌─────────────────────┐
                                     │  SYSTEM METRICS     │
                                     │  Latency            │
                                     │  Token Count / Cost │
                                     └─────────────────────┘
```

---

## 🧪 Metric API Usage

You can also use metrics **standalone** without the full Engine:

In [30]:
# Standalone metric usage example
from openagent_eval.metrics import (
    ContextPrecision, ContextRecall, MRR, NDCG, HitRate,
    PrecisionAtK, RecallAtK,
    Faithfulness, AnswerRelevancy, HallucinationDetection,
    SemanticSimilarity, ExactMatch, F1Score, BLEU, ROUGE, BERTScore,
    LatencyMetric, TokenCountMetric,
)

# Example: Compute Context Precision manually
precision_metric = ContextPrecision()

result = precision_metric.evaluate(
    question="What is RAG?",
    answer="RAG combines retrieval with generation.",
    ground_truth="RAG is a technique combining retrieval and generation.",
    contexts=[
        "RAG enhances LLMs by retrieving relevant documents.",
        "RAG combines a retriever and a generator.",
        "Unrelated document about cooking.",
    ],
    ground_truth_contexts=[
        "RAG enhances LLMs by retrieving relevant documents.",
        "RAG combines a retriever and a generator.",
    ],
)

print(f"Context Precision: {result.score:.4f}")
print(f"Reason: {result.reason}")
print(f"Metadata: {result.metadata}")

Context Precision: 0.0000
Reason: No contexts retrieved
Metadata: {'retrieved_count': 0, 'relevant_count': 0}


---

## 📈 Metric Score Interpretation Guide

| Score Range | Interpretation | Action |
|-------------|----------------|--------|
| 0.9 - 1.0 | 🟢 Excellent | Production ready |
| 0.7 - 0.9 | 🟡 Good | Minor improvements needed |
| 0.5 - 0.7 | 🟠 Fair | Investigate retrieval/generation |
| 0.3 - 0.5 | 🔴 Poor | Major issues — debug pipeline |
| 0.0 - 0.3 | 💀 Critical | Pipeline fundamentally broken |

> **Note**: For the `hallucination` metric, **lower is better** (0 = no hallucination, 1 = severe hallucination).

---

➡️ **Next: Interpreting Results & Diagnosing Failures**

# 6️⃣ Interpreting Results: Diagnose & Improve Your RAG

---

## 🔬 The Diagnostic Framework

When evaluation scores are low, use this decision tree:

```
                    ┌─────────────────────┐
                    │  Low Overall Score? │
                    └──────────┬──────────┘
                               │
              ┌────────────────┴────────────────┐
              ▼                                 ▼
    ┌─────────────────────┐           ┌─────────────────────┐
    │ Low RETRIEVAL       │           │ Low GENERATION      │
    │ metrics?            │           │ metrics?            │
    └──────────┬──────────┘           └──────────┬──────────┘
               │                                 │
      ┌────────┴────────┐               ┌────────┴────────┐
      ▼                 ▼               ▼                 ▼
  Low Precision    Low Recall      Low Faithfulness  Low Relevancy
  (noise in        (missing        (hallucination,   (off-topic,
   context)        context)        ignores context)  verbose)
      │                 │               │                 │
      ▼                 ▼               ▼                 ▼
┌─────────────┐  ┌─────────────┐ ┌─────────────┐  ┌─────────────┐
│ • Better    │  │ • Increase  │ │ • Improve   │  │ • Better    │
│   chunking  │  │   k         │ │   prompt    │  │   prompt    │
│ • Rerank    │  │ • Better    │ │ • Add       │  │ • Few-shot  │
│ • Filter    │  │   embeddings│ │   citations │  │ • Constrain │
└─────────────┘  └─────────────┘ └─────────────┘  └─────────────┘
```

---

## 🎯 Common Failure Patterns & Fixes

### Pattern 1: High Recall, Low Precision

| Symptom | Diagnosis | Fix |
|---------|-----------|-----|
| Retrieves many docs, most irrelevant | Retriever too broad | Add reranker, increase similarity threshold, improve chunking |

### Pattern 2: Low Recall, High Precision

| Symptom | Diagnosis | Fix |
|---------|-----------|-----|
| Few docs retrieved, but they're relevant | Retriever too strict / k too small | Increase k, use hybrid search (BM25 + vector), improve embeddings |

### Pattern 3: Good Retrieval, Poor Faithfulness

| Symptom | Diagnosis | Fix |
|---------|-----------|-----|
| Relevant context retrieved but answer hallucinates | LLM ignores context | Strengthen prompt ("Answer ONLY from context"), add citations, lower temperature |

### Pattern 4: Good Retrieval, Poor Relevancy

| Symptom | Diagnosis | Fix |
|---------|-----------|-----|
| Answer is factual but doesn't answer the question | Prompt too open-ended | Add explicit instructions, few-shot examples, constrain output format |

### Pattern 5: High Hallucination Score

| Symptom | Diagnosis | Fix |
|---------|-----------|-----|
| Answer contains unsupported claims | LLM overconfident | Add "I don't know" option, require citations, use smaller model with RAG |

---

## 📊 Practical Example: Analyzing Our Results

Let's write a diagnostic function to automatically flag issues:

In [31]:
# Diagnostic analysis function
def diagnose_evaluation(report, retrieval_threshold=0.7, generation_threshold=0.7):
    """Analyze an evaluation report and provide actionable diagnostics."""
    diagnostics = {
        "retrieval_issues": [],
        "generation_issues": [],
        "cost_issues": [],
        "latency_issues": [],
        "recommendations": [],
    }

    metrics = report.summary.get("metrics_summary", {})

    # Retrieval diagnostics
    retrieval_metrics = [
        "context_precision", "context_recall", "mrr",
        "ndcg", "hit_rate", "precision_at_k", "recall_at_k",
    ]
    for m in retrieval_metrics:
        if m in metrics and metrics[m] < retrieval_threshold:
            diagnostics["retrieval_issues"].append(
                f"{m}: {metrics[m]:.3f} (threshold: {retrieval_threshold})"
            )

    # Generation diagnostics
    generation_metrics = [
        "faithfulness", "answer_relevancy", "semantic_similarity", "f1_score",
    ]
    for m in generation_metrics:
        if m in metrics and metrics[m] < generation_threshold:
            diagnostics["generation_issues"].append(
                f"{m}: {metrics[m]:.3f} (threshold: {generation_threshold})"
            )

    # Hallucination (higher = worse)
    if "hallucination" in metrics and metrics["hallucination"] > 0.3:
        diagnostics["generation_issues"].append(
            f"hallucination: {metrics['hallucination']:.3f} (threshold: 0.3, lower is better)"
        )

    # Cost diagnostics
    if "token_count" in metrics and metrics["token_count"] > 2000:
        diagnostics["cost_issues"].append(
            f"High token usage: {metrics['token_count']:.0f} avg tokens/item"
        )

    # Latency diagnostics
    avg_latency = report.summary.get("average_latency_ms", 0)
    if avg_latency and avg_latency > 5000:
        diagnostics["latency_issues"].append(
            f"High latency: {avg_latency:.0f}ms avg"
        )

    # Generate recommendations
    if diagnostics["retrieval_issues"]:
        if any("precision" in i for i in diagnostics["retrieval_issues"]):
            diagnostics["recommendations"].append(
                "🔧 Add reranker (cross-encoder) to filter retrieved docs")
            diagnostics["recommendations"].append(
                "🔧 Improve chunking strategy (smaller chunks, semantic chunking)")
        if any("recall" in i or "hit_rate" in i for i in diagnostics["retrieval_issues"]):
            diagnostics["recommendations"].append(
                "🔧 Increase retrieval k (try 5-10)")
            diagnostics["recommendations"].append(
                "🔧 Try hybrid search (BM25 + vector)")
            diagnostics["recommendations"].append(
                "🔧 Upgrade embedding model (e.g., text-embedding-3-large)")

    if diagnostics["generation_issues"]:
        if any("faithfulness" in i for i in diagnostics["generation_issues"]):
            diagnostics["recommendations"].append(
                "🔧 Strengthen prompt: 'Answer ONLY from provided context'")
            diagnostics["recommendations"].append(
                "🔧 Add citation requirement in prompt")
            diagnostics["recommendations"].append(
                "🔧 Lower temperature to 0.0")
        if any("relevancy" in i for i in diagnostics["generation_issues"]):
            diagnostics["recommendations"].append(
                "🔧 Add few-shot examples to prompt")
            diagnostics["recommendations"].append(
                "🔧 Explicitly constrain answer format")
        if any("hallucination" in i for i in diagnostics["generation_issues"]):
            diagnostics["recommendations"].append(
                "🔧 Add 'I don't know' option for unsupported questions")
            diagnostics["recommendations"].append(
                "🔧 Use smaller, more controllable model")

    if diagnostics["cost_issues"]:
        diagnostics["recommendations"].append(
            "💰 Reduce context window (fewer retrieved docs)")
        diagnostics["recommendations"].append(
            "💰 Use cheaper model (e.g., GPT-4o-mini vs GPT-4o)")
        diagnostics["recommendations"].append(
            "💰 Implement prompt caching")

    if diagnostics["latency_issues"]:
        diagnostics["recommendations"].append(
            "⚡ Enable retrieval caching")
        diagnostics["recommendations"].append(
            "⚡ Use faster embedding model")
        diagnostics["recommendations"].append(
            "⚡ Consider async/parallel retrieval")

    return diagnostics


# Run diagnostics on our report
diagnostics = diagnose_evaluation(report)

print("=" * 60)
print("🔍 DIAGNOSTIC REPORT")
print("=" * 60)

for category, issues in diagnostics.items():
    if issues:
        print(f"\n{category.upper().replace('_', ' ')}:")
        for issue in issues:
            print(f"  ⚠️  {issue}")
    else:
        print(f"\n{category.upper().replace('_', ' ')}: ✅ All clear")

🔍 DIAGNOSTIC REPORT

RETRIEVAL ISSUES:
  ⚠️  context_precision: 0.000 (threshold: 0.7)
  ⚠️  context_recall: 0.000 (threshold: 0.7)
  ⚠️  mrr: 0.000 (threshold: 0.7)
  ⚠️  ndcg: 0.000 (threshold: 0.7)
  ⚠️  hit_rate: 0.000 (threshold: 0.7)
  ⚠️  precision_at_k: 0.000 (threshold: 0.7)
  ⚠️  recall_at_k: 0.000 (threshold: 0.7)

GENERATION ISSUES:
  ⚠️  faithfulness: 0.687 (threshold: 0.7)
  ⚠️  answer_relevancy: 0.446 (threshold: 0.7)
  ⚠️  semantic_similarity: 0.600 (threshold: 0.7)
  ⚠️  hallucination: 0.313 (threshold: 0.3, lower is better)

COST ISSUES: ✅ All clear

LATENCY ISSUES: ✅ All clear

RECOMMENDATIONS:
  ⚠️  🔧 Add reranker (cross-encoder) to filter retrieved docs
  ⚠️  🔧 Improve chunking strategy (smaller chunks, semantic chunking)
  ⚠️  🔧 Increase retrieval k (try 5-10)
  ⚠️  🔧 Try hybrid search (BM25 + vector)
  ⚠️  🔧 Upgrade embedding model (e.g., text-embedding-3-large)
  ⚠️  🔧 Strengthen prompt: 'Answer ONLY from provided context'
  ⚠️  🔧 Add citation requirement in pro

---

## 📈 Per-Item Failure Analysis

OpenAgent Eval tracks **per-item errors** — not just aggregate scores. Let's examine failed items:

In [32]:
# Analyze per-item failures
print("=" * 60)
print("PER-ITEM FAILURE ANALYSIS")
print("=" * 60)

for i, result in enumerate(report.result.results):
    failed_metrics = [m for m, s in result.metrics.items() if s < 0.5]
    error_metrics = result.metadata.get("metric_errors", {})

    if failed_metrics or error_metrics:
        print(f"\nItem {i + 1}: {result.question[:50]}...")
        if failed_metrics:
            print(f"  Low-score metrics: {', '.join(failed_metrics)}")
        if error_metrics:
            for metric, error in error_metrics.items():
                print(f"  ❌ {metric} ERROR: {error}")
    else:
        print(f"\nItem {i + 1}: ✅ All metrics above threshold")

PER-ITEM FAILURE ANALYSIS

Item 1: What is RAG (Retrieval-Augmented Generation)?...
  Low-score metrics: context_precision, context_recall, mrr, ndcg, hit_rate, precision_at_k, recall_at_k, hallucination

Item 2: What is the difference between precision and recal...
  Low-score metrics: context_precision, context_recall, mrr, ndcg, hit_rate, precision_at_k, recall_at_k, answer_relevancy, hallucination, semantic_similarity, bertscore
  ❌ semantic_similarity ERROR: Score must be between 0.0 and 1.0, got 1.0000001192092896
  ❌ bertscore ERROR: Score must be between 0.0 and 1.0, got 1.0000001192092896

Item 3: What are vector embeddings and how are they used i...
  Low-score metrics: context_precision, context_recall, mrr, ndcg, hit_rate, precision_at_k, recall_at_k, hallucination

Item 4: What is a chunking strategy in RAG and why does it...
  Low-score metrics: context_precision, context_recall, mrr, ndcg, hit_rate, precision_at_k, recall_at_k, answer_relevancy, hallucination

Item 5: Wh

---

## 🔄 Iterative Improvement Workflow

```
┌─────────────┐    ┌─────────────┐    ┌─────────────┐    ┌─────────────┐
│  EVALUATE   │───▶│  DIAGNOSE   │───▶│   FIX       │───▶│  RE-EVALUATE│
│  (baseline) │    │  (root      │    │  (retriever,│    │  (verify    │
│             │    │   cause)    │    │   prompt,   │    │   fix)      │
└─────────────┘    └─────────────┘    │   model)    │    └─────────────┘
       ▲                                 └─────────────┘            │
       │                                        │                    │
       └────────────────────────────────────────┘                    │
                    (repeat until scores meet thresholds)           │
```

---

➡️ **Next: Advanced Usage Patterns**

# 7️⃣ Advanced Usage

---

## 🔧 Custom Metrics

OpenAgent Eval's plugin system lets you add **custom metrics**. Here's how to create one:

In [33]:
# Custom Metric Example: Citation Accuracy
# Checks if the answer cites sources in [1], [2] format
import re
from openagent_eval.metrics.base import BaseMetric, MetricResult


class CitationAccuracyMetric(BaseMetric):
    """Evaluates whether the answer includes proper citations."""

    name = "citation_accuracy"
    description = "Measures if generated answers cite sources using [n] format"

    def evaluate(self, **kwargs):
        answer = kwargs.get("answer", "")
        contexts = kwargs.get("contexts", [])

        if not answer:
            return MetricResult(score=0.0, reason="Empty answer")

        # Find all citations like [1], [2], etc.
        citations = re.findall(r"\[(\d+)\]", answer)

        if not citations:
            return MetricResult(
                score=0.0,
                reason="No citations found in answer",
                metadata={"citations_found": 0, "contexts_available": len(contexts)},
            )

        # Check if cited indices are valid
        valid_citations = 0
        for cite in citations:
            idx = int(cite) - 1  # convert to 0-based
            if 0 <= idx < len(contexts):
                valid_citations += 1

        score = valid_citations / len(citations) if citations else 0.0

        return MetricResult(
            score=score,
            reason=f"{valid_citations}/{len(citations)} citations valid",
            metadata={
                "citations_found": len(citations),
                "valid_citations": valid_citations,
                "contexts_available": len(contexts),
            },
        )


# Test the custom metric
citation_metric = CitationAccuracyMetric()

test_answer = "RAG combines retrieval [1] and generation [2] for better answers."
test_contexts = ["RAG retrieves docs", "RAG generates answers", "Unrelated"]

result = citation_metric.evaluate(answer=test_answer, contexts=test_contexts)

print(f"Citation Accuracy: {result.score:.2f}")
print(f"Reason: {result.reason}")
print(f"Metadata: {result.metadata}")

Citation Accuracy: 1.00
Reason: 2/2 citations valid
Metadata: {'citations_found': 2, 'valid_citations': 2, 'contexts_available': 3}


---

## 📦 Register Custom Metrics

You can register custom metrics in two ways:

### Option 1: Programmatic Registration

```python
from openagent_eval.metrics import METRIC_REGISTRY

# Add to registry
METRIC_REGISTRY["citation_accuracy"] = CitationAccuracyMetric

# Now use in config:
metrics:
  generation:
    - citation_accuracy
```

### Option 2: Plugin Entry Point (for distribution)

Create a package with `pyproject.toml`:

```toml
[project.entry-points."openagent_eval.metrics"]
citation_accuracy = "my_package.metrics:CitationAccuracyMetric"
```

---

## 🔄 Batch Evaluation

Evaluate multiple configurations efficiently:

In [34]:
# Batch evaluation: Compare multiple retriever configurations
from openagent_eval.config.models import (
    Config, RetrieverConfig, LLMConfig, MetricsConfig, ReportConfig, OutputFormat,
)
from openagent_eval.core.engine import Engine
from openagent_eval.providers.retrievers.mock import MockRetriever


async def evaluate_with_config(name, retriever, k):
    """Run evaluation with a specific retriever configuration."""
    config = Config(
        dataset=DatasetConfig(path="data/sample_questions.json", format="json"),
        llm=LLMConfig(provider="mock", model="mock-model", temperature=0.0),
        retriever=RetrieverConfig(provider="mock", settings={"k": k}),
        metrics=MetricsConfig(
            retrieval=["context_precision", "context_recall", "mrr", "hit_rate"],
            generation=["faithfulness", "answer_relevancy"],
            performance=["latency"],
            cost=["token_count"],
        ),
        report=ReportConfig(output=OutputFormat.JSON, output_dir="./reports"),
        parallel=True,
        max_workers=4,
    )

    engine = Engine(config=config, retriever=retriever, llm=mock_llm)
    report = await engine.run(EVAL_DATASET)

    metrics = report.summary.get("metrics_summary", {})
    return {
        "name": name,
        "k": k,
        "context_precision": metrics.get("context_precision", 0),
        "context_recall": metrics.get("context_recall", 0),
        "mrr": metrics.get("mrr", 0),
        "hit_rate": metrics.get("hit_rate", 0),
        "faithfulness": metrics.get("faithfulness", 0),
        "answer_relevancy": metrics.get("answer_relevancy", 0),
        "avg_latency_ms": report.summary.get("average_latency_ms", 0),
        "total_tokens": report.summary.get("total_tokens", 0),
    }


async def run_batch_comparison():
    results = []
    for k in [1, 3, 5, 10]:
        # Mock retriever that returns ground-truth contexts for testing
        mock_retriever = MockRetriever(collection_name=f"test_k{k}")
        result = await evaluate_with_config(f"k={k}", mock_retriever, k)
        results.append(result)
    return results


batch_results = await run_batch_comparison()

# Display comparison table
header = (
    f"{'Config':<10} {'Prec':>6} {'Recall':>6} {'MRR':>6} {'Hit@K':>6} "
    f"{'Faith':>6} {'Rel':>6} {'Latency':>8} {'Tokens':>8}"
)
print(header)
print("-" * 70)
for r in batch_results:
    print(
        f"{r['name']:<10} {r['context_precision']:>6.3f} "
        f"{r['context_recall']:>6.3f} {r['mrr']:>6.3f} {r['hit_rate']:>6.3f} "
        f"{r['faithfulness']:>6.3f} {r['answer_relevancy']:>6.3f} "
        f"{r['avg_latency_ms']:>8.1f} {r['total_tokens']:>8}"
    )

Config       Prec Recall    MRR  Hit@K  Faith    Rel  Latency   Tokens
----------------------------------------------------------------------
k=1         1.000  0.333  1.000  1.000  0.373  0.446      0.0      287
k=3         1.000  1.000  1.000  1.000  0.505  0.446      0.0      496
k=5         1.000  1.000  1.000  1.000  0.505  0.446      0.0      496
k=10        1.000  1.000  1.000  1.000  0.505  0.446      0.0      496


---

## 📊 Experiment Comparison

OpenAgent Eval can **compare two experiment runs** side-by-side:

In [28]:
# Compare two experiments using ComparisonReport
from openagent_eval.reports import ComparisonReport


async def run_experiment_2():
    config2 = Config(
        dataset=DatasetConfig(path="data/sample_questions.json", format="json"),
        llm=LLMConfig(provider="mock", model="mock-model", temperature=0.0),
        retriever=RetrieverConfig(provider="mock", settings={"k": 5}),
        metrics=MetricsConfig(
            retrieval=["context_precision", "context_recall", "mrr"],
            generation=["faithfulness", "answer_relevancy"],
            performance=["latency"],
            cost=["token_count"],
        ),
        report=ReportConfig(output=OutputFormat.JSON, output_dir="./reports"),
        parallel=True,
        max_workers=4,
    )
    mock_retriever_2 = MockRetriever(collection_name="experiment_2")
    engine2 = Engine(config=config2, retriever=mock_retriever_2, llm=mock_llm)
    return await engine2.run(EVAL_DATASET)


report2 = await run_experiment_2()

# Build an ExperimentComparison and generate the comparison report
from openagent_eval.reports.base import ExperimentComparison

comparison_data = ExperimentComparison(
    baseline_name="Experiment 1 (k=3)",
    experiment_name="Experiment 2 (k=5)",
    baseline_results=report.result,
    experiment_results=report2.result,
)

comparison = ComparisonReport()
comparison_output = comparison.generate(comparison_data)
print(comparison_output)

  Experiment Comparison Report

  Generated: 2026-07-10 10:40:03 UTC
  Baseline:  Experiment 1 (k=3)
  Experiment: Experiment 2 (k=5)

METRIC COMPARISON
------------------------------------------------------------
  Metric                           Baseline Experiment      Delta
  ----------------------------------------------------------
  answer_relevancy                   0.4460     0.4460 =  +0.0000
  bertscore                          0.6000     0.0000   -0.6000
  bleu                               1.0000     0.0000   -1.0000
  context_precision                  0.0000     1.0000 +  +1.0000
  context_recall                     0.0000     1.0000 +  +1.0000
  exact_match                        1.0000     0.0000   -1.0000
  f1_score                           1.0000     0.0000   -1.0000
  faithfulness                       0.6870     0.5051   -0.1819
  hallucination                      0.3130     0.0000   -0.3130
  hit_rate                           0.0000     0.0000 =  +0.0000
  lat

---

## 🎯 Custom Judges (LLM-as-a-Judge)

For nuanced evaluation, you can create **custom LLM judges**:

In [35]:
# Custom LLM Judge Example
import re
from openagent_eval.metrics.base import BaseMetric, MetricResult
from openagent_eval.providers.base.llm import LLMProvider


class CustomLLMJudge(BaseMetric):
    """A metric that uses an LLM to judge answer quality."""

    name = "custom_llm_judge"
    description = "Uses an LLM to evaluate answer quality on a custom rubric"

    def __init__(self, judge_llm, rubric):
        self.judge_llm = judge_llm
        self.rubric = rubric

    async def evaluate(self, **kwargs):
        question = kwargs.get("question", "")
        answer = kwargs.get("answer", "")
        ground_truth = kwargs.get("ground_truth", "")
        contexts = kwargs.get("contexts", [])

        prompt = (
            f"{self.rubric}\n\n"
            f"Question: {question}\n"
            f"Ground Truth: {ground_truth}\n"
            f"Answer: {answer}\n"
            f"Contexts: {contexts}\n\n"
            "Score (0-1): "
        )

        response = await self.judge_llm.generate(prompt)

        # Parse score from response
        score_match = re.search(r"(\d\.?\d*)", response)
        score = float(score_match.group(1)) if score_match else 0.0
        score = min(max(score, 0.0), 1.0)  # clamp to [0, 1]

        return MetricResult(
            score=score,
            reason=f"LLM Judge score: {score}",
            metadata={"judge_response": response},
        )


# Example rubric
RUBRIC = (
    "You are an expert evaluator. Rate the answer from 0 to 1 based on:\n"
    "1. Accuracy: Does the answer correctly address the question?\n"
    "2. Completeness: Does it cover all important aspects?\n"
    "3. Grounding: Is it supported by the provided contexts?\n"
    "4. Conciseness: Is it free of unnecessary fluff?\n\n"
    "Output ONLY a number between 0 and 1."
)

print("Custom LLM Judge metric defined. Use with a real LLM provider for production.")

Custom LLM Judge metric defined. Use with a real LLM provider for production.


---

## 📈 Experiment Tracking Integration

OpenAgent Eval's JSON reports integrate with experiment tracking tools:

```python
# Log to MLflow
import mlflow

with mlflow.start_run():
    mlflow.log_params({
        "llm_provider": config.llm.provider,
        "llm_model": config.llm.model,
        "retriever_provider": config.retriever.provider,
        "k": config.retriever.settings.get("k", 5),
    })
    mlflow.log_metrics(report.summary["metrics_summary"])
    mlflow.log_metric("total_tokens", report.summary["total_tokens"])
    mlflow.log_metric("avg_latency_ms", report.summary["average_latency_ms"])

    # Log report as artifact
    mlflow.log_artifact("./reports/rag_eval_report.json")
```

---

➡️ **Next: Best Practices for Production**

# 8️⃣ Best Practices for Production RAG Evaluation

---

## 🏭 Production Evaluation Checklist

### 1. Dataset Quality

- ✅ **Curate diverse questions** — cover edge cases, ambiguous queries, multi-hop reasoning
- ✅ **Include ground-truth contexts** — not just answers; enables retrieval metrics
- ✅ **Version your datasets** — track dataset changes alongside model changes
- ✅ **Separate test/dev sets** — don't optimize on your test set
- ✅ **Minimum 50-100 questions** — statistical significance for metric averages

### 2. Metric Selection

| Use Case | Primary Metrics |
|----------|-----------------|
| **General RAG** | Faithfulness, Answer Relevancy, Context Precision, Context Recall |
| **Fact-seeking QA** | Exact Match, F1, Faithfulness, Hallucination |
| **Summarization** | ROUGE, BERTScore, Faithfulness |
| **Chat/Conversational** | Answer Relevancy, Semantic Similarity |
| **Cost-sensitive** | Token Count, Latency |
| **High-stakes (medical/legal)** | Faithfulness, Hallucination, Citation Accuracy |

### 3. Evaluation Cadence

- 🔄 **CI/CD**: Run on every PR (use mock providers for speed)
- 📅 **Nightly**: Full evaluation with real LLM on production dataset
- 🚀 **Pre-deployment**: Full eval + A/B comparison vs current production
- 📊 **Weekly**: Drift detection on production traffic (sample & evaluate)

### 4. Thresholds & Alerting

```yaml
# Example: config with thresholds
metrics:
  retrieval:
    - context_precision
    - context_recall
  generation:
    - faithfulness
    - answer_relevancy
    - hallucination
```

```python
# In your CI script:
THRESHOLDS = {
    "context_precision": 0.7,
    "context_recall": 0.7,
    "faithfulness": 0.8,
    "answer_relevancy": 0.7,
    "hallucination": 0.2,  # lower is better
}


def check_thresholds(report):
    for metric, threshold in THRESHOLDS.items():
        score = report.summary["metrics_summary"].get(metric, 0)
        if metric == "hallucination":
            passed = score <= threshold
        else:
            passed = score >= threshold
        if not passed:
            raise AssertionError(
                f"{metric}={score:.3f} below threshold {threshold}"
            )
```

### 5. Cost Optimization

- 💰 **Track tokens per query** — identify expensive outliers
- 💰 **Use cheaper models for eval** — GPT-4o-mini for eval, GPT-4o for production
- 💰 **Cache embeddings** — don't re-embed same documents
- 💰 **Limit context window** — fewer retrieved docs = lower cost
- 💰 **Batch evaluations** — amortize fixed costs

### 6. Common Mistakes to Avoid

| ❌ Mistake | ✅ Fix |
|------------|--------|
| Only evaluating generation | Always evaluate retrieval too |
| Using only Exact Match | Use semantic metrics (BERTScore, Semantic Similarity) |
| No ground-truth contexts | Include `ground_truth_contexts` in dataset |
| Evaluating on training data | Use held-out test set |
| Single metric focus | Use metric *portfolio* (retrieval + generation + cost) |
| Ignoring latency/cost | Track TokenCountMetric and LatencyMetric |
| No failure analysis | Inspect per-item errors in reports |
| Manual threshold tuning | Automate with CI gates |

### 7. Performance Tips

- ⚡ **Parallel evaluation**: Set `parallel: true`, `max_workers: 4-8`
- ⚡ **Mock providers for CI**: Use `provider: mock` for fast feedback
- ⚡ **Limit dataset size in CI**: Use `dataset.limit: 20` for quick checks
- ⚡ **Reuse vector store**: Don't rebuild embeddings every run
- ⚡ **Async throughout**: OpenAgent Eval is fully async — use `await`

---

## 📋 Production Config Template

```yaml
# production-config.yaml
dataset:
  path: data/prod_eval_set.json
  format: json
  limit: null  # full dataset
  shuffle: false

llm:
  provider: openai
  model: gpt-4o-mini
  temperature: 0.0
  max_tokens: 1024

retriever:
  provider: chroma
  settings:
    collection_name: production_docs
    k: 5
  embedder:
    provider: openai
    model: text-embedding-3-small

metrics:
  retrieval:
    - context_precision
    - context_recall
    - mrr
    - ndcg
    - hit_rate
  generation:
    - faithfulness
    - answer_relevancy
    - hallucination
    - semantic_similarity
    - f1_score
  performance:
    - latency
  cost:
    - token_count

report:
  output: json
  output_dir: ./reports/production
  include_examples: true
  max_examples: 20

parallel: true
max_workers: 8
timeout: 600.0
```

---

➡️ **Next: Conclusion & Resources**

# 9️⃣ Conclusion & Resources

---

## 🎓 What You've Learned

Congratulations! You've completed the **OpenAgent Eval RAG Evaluation Tutorial**. Here's a recap:

| Section | Key Takeaway |
|---------|--------------|
| **1. Introduction** | RAG fails in subtle ways; systematic evaluation is essential |
| **2. Installation** | `pip install openagent-eval` — zero-config, local-first |
| **3. Build RAG** | Documents → Chunking → Embeddings → Vector Store → Retriever → LLM |
| **4. Evaluate** | `Engine(config, retriever, llm).run(dataset)` — inject your real components |
| **5. Metrics** | 18 metrics: 7 retrieval, 9 generation, 1 performance, 1 cost |
| **6. Interpret** | Diagnostic framework: retrieval vs generation failures → targeted fixes |
| **7. Advanced** | Custom metrics, batch eval, experiment comparison, LLM judges |
| **8. Best Practices** | Dataset quality, metric selection, CI/CD integration, cost tracking |

---

## 🔗 Essential Links

| Resource | Link |
|----------|------|
| 📦 **Repository** | https://github.com/openagenthq/openagent-eval |
| 📚 **Documentation** | https://docs.openagenthq.com |
| 🐛 **Issues** | https://github.com/openagenthq/openagent-eval/issues |
| 💬 **Discussions** | https://github.com/openagenthq/openagent-eval/discussions |
| 🤝 **Contributing** | https://github.com/openagenthq/openagent-eval/blob/main/CONTRIBUTING.md |
| 📋 **Changelog** | https://github.com/openagenthq/openagent-eval/blob/main/CHANGELOG.md |
| 🗺️ **Roadmap** | https://github.com/openagenthq/openagent-eval/blob/main/ROADMAP.md |

---

## 🚀 Next Steps

1. **Try with your own data** — Replace the sample corpus with your documents
2. **Swap providers** — Change `mock` → `openai`/`anthropic`/`gemini`/`ollama` in config
3. **Add to CI/CD** — Run `oaeval run config.yaml` in your pipeline
4. **Create custom metrics** — Extend `BaseMetric` for domain-specific evaluation
5. **Contribute!** — We welcome PRs for new metrics, providers, and report formats

---

## 🙏 Acknowledgments

OpenAgent Eval is built by the **OpenAgentHQ** community. Special thanks to:

- **Ragas** team for faithfulness/answer_relevancy implementations
- **DeepEval** team for hallucination detection patterns
- **Hugging Face** for the `evaluate` library (BLEU, ROUGE, BERTScore)
- **Sentence Transformers** for semantic similarity embeddings
- All **contributors** who've filed issues, written docs, and submitted PRs

---

## 📝 Feedback & Questions

- Found a bug? → [Open an issue](https://github.com/openagenthq/openagent-eval/issues)
- Have a question? → [Start a discussion](https://github.com/openagenthq/openagent-eval/discussions)
- Want to contribute? → Read [CONTRIBUTING.md](https://github.com/openagenthq/openagent-eval/blob/main/CONTRIBUTING.md)
- Need help? → Check [FAQ](https://docs.openagenthq.com/faq) or [Support](https://github.com/openagenthq/openagent-eval/blob/main/SUPPORT.md)

---

```
    ╔══════════════════════════════════════════════════════════════╗
    ║                                                               ║
    ║   Happy Evaluating! 🎉                                       ║
    ║                                                               ║
    ║   Remember: The best RAG system is the one you've measured.  ║
    ║                                                               ║
    ╚══════════════════════════════════════════════════════════════╝
```